In [1]:
import json
import os
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Literal

import optuna
import wandb
from dotenv import load_dotenv

sys.path.append(os.path.abspath("../.."))

import src.utils.run_optuna as op
from src.utils.optuna_objective import create_objective
from src.utils.telegram import send_message

### Config

In [2]:
load_dotenv(dotenv_path="../../.env")


@dataclass
class Config:
    # Data / CV
    model_name: str = "ridge"
    data_id: str = "058"
    n_folds: int = 5
    seed: int = 42
    fold_idx: int = 0

    # Optuna
    n_trials: int = 10
    direction: str = "maximize"
    sampler: str = "tpe"  # tpe / random
    pruner: str = "median"  # median / none

    # Initial params
    use_initial: Literal["never", "manual"] = "never"
    initial_param_sources: list[tuple[str, int]] = field(default_factory=list)   # (study_name, n_trial) 例: ("xgb-001", 1)

    # Storage
    storage: str = "sqlite:////home/hanse/kaggle/binary-bank/artifacts/optuna/optuna.db"

    # Option
    opts: dict = field(default_factory=dict)


cfg = Config()
cfg.initial_param_sources = [("lgbm-057", 16)]

# W&B
wandb_project = os.environ.get("COMPETITION_NAME")
wandb.login(key=os.environ.get("WANDB_API_KEY"))

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/hanse/.netrc
wandb: Currently logged in as: kaitookano (kaitookano-waseda-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

### Build & Run

In [3]:
# --- helper: initial params loader ---
def load_initial_params(sources: list[tuple[str, int]]) -> list[dict]:
    loaded = []
    for study, n_trial in sources:
        path = Path(f"../../artifacts/optuna/{study}/trl{n_trial}.json")
        with path.open("r") as f:
            params = json.load(f)["params"]
        loaded.append(params)
    return loaded


# sampler / pruner factory
def build_sampler(name, seed):
    if name == "tpe":
        return optuna.samplers.TPESampler(n_startup_trials=15, seed=seed)
    elif name == "random":
        return optuna.samplers.RandomSampler(seed=seed)
    else:
        raise ValueError(f"unknown sampler: {name}")


def build_pruner(name):
    if name == "median":
        return optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1000)
    elif name == "none":
        return optuna.pruners.NopPruner()
    else:
        raise ValueError(f"unknown pruner: {name}")


objective = create_objective(
    cfg.model_name,
    cfg.data_id,
    seed=cfg.seed,
    n_folds=cfg.n_folds,
    fold_idx=cfg.fold_idx,
    wandb_project=wandb_project,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    opts=cfg.opts
)

sampler = build_sampler(cfg.sampler, cfg.seed)
pruner = build_pruner(cfg.pruner)

initial_params = None
if cfg.use_initial == "manual":
    initial_params = load_initial_params(cfg.initial_param_sources)

op.run_optuna_search(
    objective,
    n_trials=cfg.n_trials,
    direction=cfg.direction,
    study_name=f"{cfg.model_name}-{cfg.data_id}",
    storage=cfg.storage,
    sampler=sampler,
    pruner=pruner,
    initial_params=initial_params
)

[I 2025-10-06 22:29:07,331] Using an existing study with name 'ridge-058' instead of creating a new one.


[initial] none


  0%|          | 0/10 [00:00<?, ?it/s]

Fold Col: 5fold-s42
Free CPU Mem: 12.27 GB
Free GPU Mem: 5.18 GB
RMSE Valid: 0.19382
R2 Valid: 0.64590
MAE Valid: 0.07541
AUC Valid: 0.97737
Total Runtime: 00:00:01
Free CPU Mem: 11.93 GB
Free GPU Mem: 5.13 GB


auc_f1,0.97737
mae_f1,0.07541
r2_f1,0.6459
rmse_f1,0.19382
runtime_f1,0.02204


...

auc_f1,0.97737
mae_f1,0.07541
r2_f1,0.6459
rmse_f1,0.19382
runtime_f1,0.01158


[I 2025-10-06 22:29:40,821] Trial 22 finished with value: 0.9773680997933234 and parameters: {'alpha': 0.011345388315103903}. Best is trial 18 with value: 0.9773681041920608.


Fold Col: 5fold-s42
Free CPU Mem: 11.82 GB
Free GPU Mem: 5.17 GB
RMSE Valid: 0.19382
R2 Valid: 0.64590
MAE Valid: 0.07541
AUC Valid: 0.97737
Total Runtime: 00:00:00
Free CPU Mem: 11.79 GB
Free GPU Mem: 5.13 GB


auc_f1,0.97737
mae_f1,0.07541
r2_f1,0.6459
rmse_f1,0.19382
runtime_f1,0.01218


[I 2025-10-06 22:29:49,718] Trial 23 finished with value: 0.977368100002787 and parameters: {'alpha': 0.011520489947554346}. Best is trial 18 with value: 0.9773681041920608.
✅ Message sent.
